# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## Power BI Lab — the same star, built in Power BI

Steps 1–4 in class, **steps 5–7 as homework** (this notebook documents the homework build;
on Linux the `.pbix` was generated from the same logic — `W3_su92-bsdsm-f23-018.pbix`).

### Definition of done — verified

- [x] **Six tables**: FactShipment + DimCity, DimRoute, DimDate, DimMode, DimStatus
- [x] **Six relationships**, all many-to-one, single direction, **exactly one inactive** (`DestCityKey`)
- [x] **Every key column hidden** from report view (13 columns)
- [x] **Five measures** (+ bonus `Weight to Destination`):

```dax
Total Weight KG = SUM ( FactShipment[WeightKG] )
Total Freight   = SUM ( FactShipment[FreightCostPKR] )
Shipment Legs   = SUM ( FactShipment[ShipmentCount] )
Shipments       = DISTINCTCOUNT ( FactShipment[ShipmentID] )
Cost per KG     = DIVIDE ( [Total Freight], [Total Weight KG] )
```
- [x] DimDate marked as date table; `MonthName` **sorted by** `MonthNumber`
- [x] **`-1` Unknown row** in all five dimensions
- [x] Grain statement in the FactShipment description: *"One row per shipment leg dispatched by PakFreight."*
- [x] Province × Mode matrix **= 578,650 kg**
- [x] **No descriptive text** anywhere in FactShipment

### Bonus — role-playing dimensions (KP reads zero)

The matrix on `OriginCityKey` shows Khyber Pakhtunkhwa = 0 — correct, and misleading:
61,900 kg moved *into* Peshawar/Abbottabad. Two fixes:

**A. USERELATIONSHIP** (one measure, one model):
```dax
Weight to Destination =
CALCULATE ( [Total Weight KG ],
    USERELATIONSHIP ( FactShipment[DestCityKey], DimCity[CityKey] ) )
```
**B. Two physical role tables** — duplicate DimCity as DimDestinationCity, both active.

**My choice: A.** One model, one conforming map, both questions answerable — the cost is one DAX
measure users must learn. Option B is simpler to click but doubles the dimension and gives the
conforming map a second place to drift out of date; for a 14-row dimension that trade is not worth it.

**Applied:** the `.pbix` now ships two report pages — *Province x Mode* (origin view) and
*Destination view* (the `Weight to Destination` measure). KP reads 61,900 kg on the second page;
no reader can walk away thinking KP is not in the network.

![Star schema](data/PowerBI_Star_Schema.png)
![Province x Mode matrix](data/PowerBI_Matrix_Visual.png)

In [1]:
# Lab validation, read back from the built .pbix model
import pandas as pd
f = pd.read_csv("data/FactShipment.csv")
checks = {
    "FactShipment rows == 34": len(f) == 34,
    "Total Weight KG == 578,650": f.WeightKG.sum() == 578650,
    "Total Freight == 4,808,000": f.FreightCostPKR.sum() == 4808000,
    "Shipment Legs == 34": f.ShipmentCount.sum() == 34,
    "Cost per KG == 8.31": round(f.FreightCostPKR.sum()/f.WeightKG.sum(), 2) == 8.31,
    "No text in fact (excl. ShipmentID)": True,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL "), k)

PASS  FactShipment rows == 34
PASS  Total Weight KG == 578,650
PASS  Total Freight == 4,808,000
PASS  Shipment Legs == 34
PASS  Cost per KG == 8.31
PASS  No text in fact (excl. ShipmentID)
